# 评测微实验：重复运行为什么不等于独立任务？

约 20–30 分钟。纯 Python 标准库、CPU，不调用模型。按顺序运行单元格。

用**刻意高度相关的合成数据**比较两种 bootstrap：按任务重采样和把所有运行当独立样本。这个例子说明统计单位的影响，不是任何模型或框架的性能结论，也不是所有真实数据都会呈现的差异。

先预测：40 个任务，每个重复 5 次，是否获得了 200 个独立任务的证据？

对应专题：[评测统计与消融](https://suguilin.github.io/study/?id=lesson-evaluation-statistics)。

In [ ]:
from collections import defaultdict
from pathlib import Path
import csv
import json
import random
from statistics import mean

SEED = 20261011
BOOTSTRAP_SAMPLES = 5000
USE_MY_CSV = False

def synthetic_runs():
    rows = []
    for task in range(40):
        # Repeated outcomes within a task are intentionally perfectly correlated.
        baseline, variant = (0, 1) if task < 8 else (1, 0) if task < 12 else (1, 1) if task < 26 else (0, 0)
        for run in range(5):
            rows.append({'task_id': f'task-{task:02d}', 'run': run, 'baseline': baseline, 'variant': variant})
    return rows

if USE_MY_CSV:
    # Upload runs.csv through Colab's Files panel. Pair the same task and run.
    with open('runs.csv', encoding='utf-8', newline='') as f:
        rows = [dict(row, run=int(row['run']), baseline=int(row['baseline']), variant=int(row['variant']))
                for row in csv.DictReader(f)]
else:
    rows = synthetic_runs()

assert rows, 'Provide at least one paired observation'
assert all(row['baseline'] in (0, 1) and row['variant'] in (0, 1) and row['task_id'] for row in rows)
assert len({(row['task_id'], row['run']) for row in rows}) == len(rows), 'Duplicate task/run pairs'
by_task = defaultdict(list)
for row in rows:
    by_task[row['task_id']].append(row['variant'] - row['baseline'])
task_deltas = [mean(values) for values in by_task.values()]
flat_deltas = [row['variant'] - row['baseline'] for row in rows]
print('Tasks:', len(by_task), 'paired runs:', len(rows))
print('Per-task average change:', round(mean(task_deltas), 4))


## 1 · 保留任务内相关性

下面按任务平均差值重采样，用等权任务解释结果；这保留重复运行的分组单位。同一任务的 baseline / variant 已经配对。此处只有两个候选的比较，没有处理多重比较或生产任务分布偏移。

In [ ]:
def percentile(sorted_values, q):
    position = (len(sorted_values) - 1) * q
    lower = int(position)
    upper = min(lower + 1, len(sorted_values) - 1)
    return sorted_values[lower] + (sorted_values[upper] - sorted_values[lower]) * (position - lower)

def bootstrap(values, samples=BOOTSTRAP_SAMPLES, seed=SEED):
    rng = random.Random(seed)
    estimates = sorted(mean(rng.choices(values, k=len(values))) for _ in range(samples))
    interval = (percentile(estimates, .025), percentile(estimates, .975))
    return {'estimate': mean(values), 'lower': interval[0], 'upper': interval[1],
            'width': interval[1] - interval[0]}

grouped = bootstrap(task_deltas)
naive = bootstrap(flat_deltas)
print('Task-grouped paired bootstrap:', grouped)
print('Incorrect independent-run assumption:', naive)
if not USE_MY_CSV:
    assert abs(grouped['estimate'] - .1) < 1e-9
    assert grouped['width'] > naive['width']


## 2 · 改变重复次数，观察区间

把同一批结果机械复制 4 次，没有增加新任务。正确的任务分组区间应不受这种复制影响；独立运行的错误假设会把区间变窄。它展示的是“伪重复”的风险。

真正新增独立运行可能提供信息，但必须结合任务内随机性设计分析；不要从本合成数据推断重复运行永远没有价值。

In [ ]:
replicated_flat = flat_deltas * 4
replicated_task = [mean(values * 4) for values in by_task.values()]
replicated_grouped = bootstrap(replicated_task)
replicated_naive = bootstrap(replicated_flat)
print('Grouped after copying observations:', replicated_grouped)
print('Naive after copying observations:', replicated_naive)
assert replicated_grouped == grouped
if not USE_MY_CSV:
    assert replicated_naive['width'] < naive['width']


## 3 · 用真实数据替换，先解释协议

需要时上传 `runs.csv`，列名为 `task_id,run,baseline,variant`，两个结果都用 0 / 1。设置 `USE_MY_CSV=True` 后重新运行全部单元格。任务与运行编号必须正确配对。所有数据仅在你的 Colab runtime 中读取，本例不挂载 Drive、不上传网络。

解释题：

1. 任务分组区间包含 0 时，你可以得出哪些结论，哪些结论还不支持？
2. 等权任务与等权运行在任务重复次数不同的时候有什么区别？
3. 如果两个版本更改了模型和检索策略，成功率变化能否归因于检索？还需要哪项消融？

在下面记录你的结论；可下载报告或复制到网站练习。

In [ ]:
my_interpretation = ""  # Explain the uncertainty and what to test next.
report = {'experiment': 'paired-agent-evaluation', 'data_scope': 'user CSV' if USE_MY_CSV else 'synthetic correlated data',
          'seed': SEED, 'samples': BOOTSTRAP_SAMPLES, 'tasks': len(by_task), 'paired_runs': len(rows),
          'grouped': grouped, 'naive': naive, 'my_interpretation': my_interpretation}
Path('evaluation_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved evaluation_report.json. Keep the task set, model, scoring and versions with any real evaluation.')
